In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

In [8]:
###load dataset

In [9]:
data = pd.read_csv("sleep_encoded.csv")
data

FileNotFoundError: [Errno 2] No such file or directory: 'sleep_encoded.csv'

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
###handling missing value
data['Sleep Disorder'] = data['Sleep Disorder'].fillna('None')

In [ ]:
data['Sleep Disorder'].value_counts()

In [ ]:
print("Total missing values:", data.isnull().sum().sum())

In [ ]:
####Separate features and target


In [ ]:
x = data.drop('Sleep Disorder', axis=1)

y = data['Sleep Disorder']

In [ ]:
print("Features shape:", x.shape)
print("Target shape:", y.shape)

In [ ]:
print("Target distribution:")
print(y.value_counts())

In [ ]:
###Visualise class distribution

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    x=y,
    order=y.value_counts().index
)

plt.title('Sleep Disorder Class Distribution')
plt.xlabel('Sleep Disorder')
plt.ylabel('Count')

plt.show()

In [ ]:
###train and test 

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
print("Training Features:", x_train.shape)
print("Testing Features:", x_test.shape)

print("\nTraining Target:")
print(y_train.value_counts())

print("\nTesting Target:")
print(y_test.value_counts())

In [ ]:
#feature scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
numeric_features = [
    'Age',
    'Sleep Duration',
    'Quality of Sleep',
    'Physical Activity Level',
    'Stress Level',
    'Heart Rate',
    'Daily Steps'
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), numeric_features)
    ],
    remainder='passthrough'
)

In [ ]:
###Model training and comparison


In [ ]:
models = {

    'Logistic Regression': LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    'SVM': SVC(),

    'KNN': KNeighborsClassifier(),

    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42
    )

}

In [ ]:
results = []
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(x_train, y_train)

    y_pred = pipeline.predict(x_test)

    results.append({
        'Model': name,
        'Balancing': 'Without Balancing',
        'Accuracy': accuracy_score(y_test, y_pred) * 100,
        'Precision': precision_score(
            y_test, y_pred,
            average='macro',
            zero_division=0
        ) * 100,
        'Recall': recall_score(
            y_test, y_pred,
            average='macro',
            zero_division=0
        ) * 100,
        'F1 Score': f1_score(
            y_test, y_pred,
            average='macro',
            zero_division=0
        ) * 100
    })

    trained_models[name] = pipeline

    print(name, "completed!")

In [ ]:
balanced_models = {}

for name, model in models.items():

    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('sampler', RandomOverSampler(random_state=42)),
        ('model', model)
    ])

    pipeline.fit(x_train, y_train)

    y_pred = pipeline.predict(x_test)

    results.append({
        'Model': name,
        'Balancing': 'Random Oversampling',
        'Accuracy': accuracy_score(y_test, y_pred) * 100,
        'Precision': precision_score(
            y_test, y_pred,
            average='macro',
            zero_division=0
        ) * 100,
        'Recall': recall_score(
            y_test, y_pred,
            average='macro',
            zero_division=0
        ) * 100,
        'F1 Score': f1_score(
            y_test, y_pred,
            average='macro',
            zero_division=0
        ) * 100
    })

    balanced_models[name] = pipeline

    print(name, "completed with balancing!")

In [10]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1 Score',
    ascending=False
).reset_index(drop=True)

results_df.round(2)

NameError: name 'results' is not defined

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_df,
    x='Model',
    y='F1 Score',
    hue='Balancing'
)

plt.title('Model Performance Comparison')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
###testing svm

In [ ]:
svm_model = Pipeline([
    ('preprocessor', preprocessor),

    ('model', SVC(
        kernel='rbf',
        C=1,
        class_weight='balanced',
        random_state=42
    ))
])

In [ ]:
svm_model.fit(x_train, y_train)

y_pred_svm = svm_model.predict(x_test)

In [ ]:
print(
    "SVM Accuracy:",
    accuracy_score(y_test, y_pred_svm)
)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

In [ ]:
##testing random forest

In [ ]:
rf_model = Pipeline([
    ('preprocessor', preprocessor),

    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42
    ))
])

In [ ]:
rf_model.fit(x_train, y_train)

y_pred_rf = rf_model.predict(x_test)

In [ ]:
print(
    "Random Forest Accuracy:",
    accuracy_score(y_test, y_pred_rf)
)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
##cross validation

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
cv_scores = cross_val_score(
    svm_model,
    x_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

In [ ]:
print("Cross-validation scores:", cv_scores)

print("\nMean CV Accuracy:", cv_scores.mean())

print("Standard Deviation:", cv_scores.std())

In [ ]:
###Hyperparameter tuning using GridSearchCV

In [ ]:
param_grid = {

    'model__C': [0.1, 1, 10, 100],

    'model__kernel': ['rbf', 'linear'],

    'model__gamma': ['scale', 'auto'],

    'model__class_weight': ['balanced']

}

In [ ]:
grid_search = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(x_train, y_train)

In [ ]:
###Evaluate tuned SVM

In [ ]:
best_svm_model = grid_search.best_estimator_

y_pred_tuned = best_svm_model.predict(x_test)

In [ ]:
print(
    "Tuned SVM Accuracy:",
    accuracy_score(y_test, y_pred_tuned)
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_tuned
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_tuned,
    labels=best_svm_model.classes_
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=best_svm_model.classes_,
    yticklabels=best_svm_model.classes_
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Tuned SVM Confusion Matrix")

plt.show()

In [ ]:
#####Check overfitting and underfitting

In [ ]:
y_train_pred = best_svm_model.predict(x_train)

train_accuracy = accuracy_score(
    y_train,
    y_train_pred
)

test_accuracy = accuracy_score(
    y_test,
    y_pred_tuned
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='macro'
)

test_f1 = f1_score(
    y_test,
    y_pred_tuned,
    average='macro'
)

print("Training Accuracy:", train_accuracy * 100)

print("Testing Accuracy:", test_accuracy * 100)

print("\nTraining F1 Score:", train_f1 * 100)

print("Testing F1 Score:", test_f1 * 100)

In [ ]:
####Select the final model

In [11]:
final_model = best_svm_model

print("Final Model Selected: Tuned SVM")

NameError: name 'best_svm_model' is not defined

In [ ]:
final_predictions = final_model.predict(x_test)

final_accuracy = accuracy_score(
    y_test,
    final_predictions
)

print("Final Model Accuracy:", final_accuracy * 100)

In [ ]:
print(
    classification_report(
        y_test,
        final_predictions
    )
)

In [ ]:
os.makedirs('../models', exist_ok=True)

In [ ]:
joblib.dump(
    final_model,
    '../models/svm_sleep_disorder_model.pkl'
)

print("Final tuned SVM model saved successfully!")

In [ ]:
test_predictions = loaded_model.predict(x_test)

print(
    "Loaded Model Accuracy:",
    accuracy_score(y_test, test_predictions) * 100
)